# Entrenamiento de modelos YOLO para detección de defectos en PCB

Este notebook implementa el entrenamiento de modelos YOLO para la detección automática de defectos en placas de circuito impreso.

Se comparan dos variantes de la misma familia de modelos:

- **YOLO11 Nano (`yolo11n`)**: variante de menor tamaño y costo computacional.
- **YOLO11 Small (`yolo11s`)**: variante de mayor capacidad representacional y costo computacional.

Ambos modelos se entrenan mediante *fine-tuning* a partir de pesos preentrenados, utilizando las mismas particiones de entrenamiento y validación y una configuración experimental comparable.

## Objetivos

- Realizar *fine-tuning* de YOLO11 Nano sobre DeepPCB.
- Realizar *fine-tuning* de YOLO11 Small sobre DeepPCB.
- Mantener condiciones experimentales comparables entre ambos modelos.
- Registrar métricas y artefactos de entrenamiento.
- Analizar la convergencia de los modelos.
- Comparar desempeño y costo computacional.
- Reservar la partición `test` exclusivamente para la evaluación final.

## 1. Importación de librerías y configuración

Se importan las dependencias necesarias y se definen las rutas del dataset procesado y de los resultados experimentales.

La partición `test` no será utilizada durante el entrenamiento ni para la selección de modelos.

In [1]:
from pathlib import Path
import sys

import torch
import ultralytics
from ultralytics import YOLO

In [2]:
# Detectar raíz del proyecto
PROJECT_ROOT = Path.cwd().parent

# Dataset procesado
YOLO_DATASET_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "deep_pcb_yolo"
)

DATA_YAML_PATH = YOLO_DATASET_DIR / "data.yaml"

# Directorio de resultados
RUNS_DIR = PROJECT_ROOT / "runs"

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Dataset YOLO: {YOLO_DATASET_DIR}")
print(f"Configuración: {DATA_YAML_PATH}")
print(f"Resultados: {RUNS_DIR}")

Raíz del proyecto: c:\Users\zakiea\Desktop\DL\TP Final\pcb-defect-detection-yolo
Dataset YOLO: c:\Users\zakiea\Desktop\DL\TP Final\pcb-defect-detection-yolo\data\processed\deep_pcb_yolo
Configuración: c:\Users\zakiea\Desktop\DL\TP Final\pcb-defect-detection-yolo\data\processed\deep_pcb_yolo\data.yaml
Resultados: c:\Users\zakiea\Desktop\DL\TP Final\pcb-defect-detection-yolo\runs


In [3]:
assert YOLO_DATASET_DIR.exists(), (
    "No se encontró el dataset procesado."
)

assert DATA_YAML_PATH.exists(), (
    "No se encontró data.yaml."
)

print("Rutas validadas correctamente.")

Rutas validadas correctamente.


## 2. Verificación del entorno de ejecución

Se verifica la disponibilidad de aceleración por GPU y las versiones de las principales librerías utilizadas.

Esta información resulta relevante para documentar la reproducibilidad de los experimentos y analizar posteriormente el costo computacional de cada variante del modelo.

In [4]:
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Ultralytics: {ultralytics.__version__}")

print(f"\nCUDA disponible: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Versión CUDA: {torch.version.cuda}")
else:
    print("Dispositivo de entrenamiento disponible: CPU")

Python: 3.11.15
PyTorch: 2.12.1+cpu
Ultralytics: 8.4.87

CUDA disponible: False
Dispositivo de entrenamiento disponible: CPU
